In [9]:
import pandas as pd
import numpy as np
import matplotlib

# [FIX] Use non-interactive backend for stability
matplotlib.use('Agg') 
import matplotlib.pyplot as plt

# --- Configuration: File Paths ---
# 1. SBSP
PATH_SBSP = '/media/kai/NewDisk/Kai_thesis/Master_Thesis_E2E_RL_Teleop/libfranka_ws/src/E2E_Teleoperation/E2E_Teleoperation/evaluation/SBSP/low_delay_cleaned.csv'

# 2. ASAC
PATH_ASAC = '/media/kai/NewDisk/Kai_thesis/Master_Thesis_E2E_RL_Teleop/libfranka_ws/src/E2E_Teleoperation/E2E_Teleoperation/evaluation/ASAC/low_delay.csv'

# 3. PD Controller
PATH_PD   = '/media/kai/NewDisk/Kai_thesis/Master_Thesis_E2E_RL_Teleop/libfranka_ws/src/E2E_Teleoperation/E2E_Teleoperation/evaluation/PD_controller/low_delay_cleaned.csv'

# 4. DRT-RL
PATH_DRT  = '/media/kai/NewDisk/Kai_thesis/Master_Thesis_E2E_RL_Teleop/libfranka_ws/src/E2E_Teleoperation/E2E_Teleoperation/evaluation/DRT_RL/low_delay_cleaned.csv'

# 5. E2E-RL
PATH_E2E  = '/media/kai/NewDisk/Kai_thesis/Master_Thesis_E2E_RL_Teleop/libfranka_ws/src/E2E_Teleoperation/E2E_Teleoperation/evaluation/data/low_delay_cleaned.csv'

# Analysis Settings
MAX_STEPS = 5000
TARGET_DT = 0.02  # Target 50Hz

def load_and_calc_error(file_path, label):
    try:
        df = pd.read_csv(file_path)
        
        # --- Auto-Downsampling Logic ---
        if 'time' in df.columns:
            df = df.sort_values('time').reset_index(drop=True)
            dt_original = df['time'].diff().median()
            
            if dt_original is not None and dt_original > 0:
                step_ratio = int(np.round(TARGET_DT / dt_original))
                if step_ratio > 1:
                    print(f"[{label}] Downsampling by factor {step_ratio}.")
                    df = df.iloc[::step_ratio].reset_index(drop=True)

        # --- Column Validation ---
        required = ['leader_ee_pos_x', 'leader_ee_pos_y', 'leader_ee_pos_z',
                    'follower_ee_pos_x', 'follower_ee_pos_y', 'follower_ee_pos_z']
        
        if not all(col in df.columns for col in required):
            print(f"[{label}] Error: Missing standard columns.")
            return None

        # --- Truncation & Error Calculation ---
        df = df.head(MAX_STEPS)
        leader_pos = df[['leader_ee_pos_x', 'leader_ee_pos_y', 'leader_ee_pos_z']].to_numpy()
        follower_pos = df[['follower_ee_pos_x', 'follower_ee_pos_y', 'follower_ee_pos_z']].to_numpy()
        error_norm = np.linalg.norm(leader_pos - follower_pos, axis=1)
        
        return df.index.to_numpy(), error_norm
        
    except FileNotFoundError:
        print(f"[{label}] Error: File not found.")
        return None
    except Exception as e:
        print(f"[{label}] Unexpected error: {e}")
        return None

def plot_final_comparison(data_map):
    """
    Plots tracking error + average lines + focused view.
    """
    plt.figure(figsize=(12, 6))
    
    # Define Styles for consistency
    styles = {
        'PD':   {'label': 'PD Controller', 'c': 'gray',    'ls': ':',  'lw': 2,   'alpha': 0.5},
        'SBSP': {'label': 'SBSP',          'c': '#ff7f0e', 'ls': '--', 'lw': 1.5, 'alpha': 0.9},
        'ASAC': {'label': 'ASAC',          'c': '#1f77b4', 'ls': '-',  'lw': 1.5, 'alpha': 0.9},
        'DRT':  {'label': 'DRT-RL',        'c': '#2ca02c', 'ls': '-',  'lw': 1.5, 'alpha': 0.9},
        'E2E':  {'label': 'E2E-RL',        'c': '#d62728', 'ls': '-',  'lw': 1.5, 'alpha': 0.9}
    }

    # Iterate through models and plot
    for key, val in data_map.items():
        if val is not None:
            steps, errors = val
            mean_err = np.mean(errors)
            s = styles[key]
            
            # 1. Plot Trajectory with Mean in Label
            # Label format: "Name (μ=0.025m)"
            label_str = f"{s['label']} ($\mu$={mean_err:.3f}m)"
            plt.plot(steps, errors, label=label_str, 
                     color=s['c'], linestyle=s['ls'], linewidth=s['lw'], alpha=s['alpha'])
            
            # 2. Plot Horizontal Mean Line
            # We skip PD line if it's off-screen (>0.15) to keep plot clean
            if mean_err < 0.15:
                plt.axhline(y=mean_err, color=s['c'], linestyle='--', linewidth=1.0, alpha=0.6)

    # Formatting
    plt.title('Control Strategy Comparison: Low Delay Scenario (Focused View)', fontsize=14)
    plt.xlabel('Step', fontsize=12)
    plt.ylabel(r'Tracking Error $\|\mathbf{e}\|$ (m)', fontsize=12)
    
    # Focused Y-axis Limit
    plt.ylim(0, 0.15)
    
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.legend(loc='upper right', fontsize=11, framealpha=0.95)
    plt.tight_layout()
    plt.margins(x=0.01)
    
    # Save
    out_file = 'comparison_low_delay_focused_means.png'
    plt.savefig(out_file, dpi=300)
    print(f"Comparison plot saved successfully to: {out_file}")
    plt.close()

if __name__ == "__main__":
    print("Loading datasets...")
    
    # Load data into a dictionary for easier iteration
    data = {
        'PD':   load_and_calc_error(PATH_PD,   "PD"),
        'SBSP': load_and_calc_error(PATH_SBSP, "SBSP"),
        'ASAC': load_and_calc_error(PATH_ASAC, "ASAC"),
        'DRT':  load_and_calc_error(PATH_DRT,  "DRT-RL"),
        'E2E':  load_and_calc_error(PATH_E2E,  "E2E-RL")
    }
    
    plot_final_comparison(data)

Loading datasets...
Comparison plot saved successfully to: comparison_low_delay_focused_means.png


In [12]:
import pandas as pd
import numpy as np
import matplotlib

# [FIX] Use non-interactive backend for stability
matplotlib.use('Agg') 
import matplotlib.pyplot as plt

# --- Configuration: File Paths ---
# 1. SBSP
PATH_SBSP = '/media/kai/NewDisk/Kai_thesis/Master_Thesis_E2E_RL_Teleop/libfranka_ws/src/E2E_Teleoperation/E2E_Teleoperation/evaluation/SBSP/high_delay_cleaned.csv'

# 2. ASAC
PATH_ASAC = '/media/kai/NewDisk/Kai_thesis/Master_Thesis_E2E_RL_Teleop/libfranka_ws/src/E2E_Teleoperation/E2E_Teleoperation/evaluation/ASAC/high_delay.csv'

# 3. PD Controller
PATH_PD   = '/media/kai/NewDisk/Kai_thesis/Master_Thesis_E2E_RL_Teleop/libfranka_ws/src/E2E_Teleoperation/E2E_Teleoperation/evaluation/PD_controller/high_delay_cleaned.csv'

# 4. DRT-RL
PATH_DRT  = '/media/kai/NewDisk/Kai_thesis/Master_Thesis_E2E_RL_Teleop/libfranka_ws/src/E2E_Teleoperation/E2E_Teleoperation/evaluation/DRT_RL/high_delay_cleaned.csv'

# 5. E2E-RL
PATH_E2E  = '/media/kai/NewDisk/Kai_thesis/Master_Thesis_E2E_RL_Teleop/libfranka_ws/src/E2E_Teleoperation/E2E_Teleoperation/evaluation/data/high_delay_cleaned.csv'

# Analysis Settings
MAX_STEPS = 5000
TARGET_DT = 0.02  # Target 50Hz

def load_and_calc_error(file_path, label):
    try:
        df = pd.read_csv(file_path)
        
        # --- Auto-Downsampling Logic ---
        if 'time' in df.columns:
            df = df.sort_values('time').reset_index(drop=True)
            dt_original = df['time'].diff().median()
            
            if dt_original is not None and dt_original > 0:
                step_ratio = int(np.round(TARGET_DT / dt_original))
                if step_ratio > 1:
                    print(f"[{label}] Downsampling by factor {step_ratio}.")
                    df = df.iloc[::step_ratio].reset_index(drop=True)

        # --- Column Validation ---
        required = ['leader_ee_pos_x', 'leader_ee_pos_y', 'leader_ee_pos_z',
                    'follower_ee_pos_x', 'follower_ee_pos_y', 'follower_ee_pos_z']
        
        if not all(col in df.columns for col in required):
            print(f"[{label}] Error: Missing standard columns.")
            return None

        # --- Truncation & Error Calculation ---
        df = df.head(MAX_STEPS)
        leader_pos = df[['leader_ee_pos_x', 'leader_ee_pos_y', 'leader_ee_pos_z']].to_numpy()
        follower_pos = df[['follower_ee_pos_x', 'follower_ee_pos_y', 'follower_ee_pos_z']].to_numpy()
        error_norm = np.linalg.norm(leader_pos - follower_pos, axis=1)
        
        return df.index.to_numpy(), error_norm
        
    except FileNotFoundError:
        print(f"[{label}] Error: File not found.")
        return None
    except Exception as e:
        print(f"[{label}] Unexpected error: {e}")
        return None

def plot_final_comparison(data_map):
    """
    Plots tracking error + average lines + focused view.
    """
    plt.figure(figsize=(12, 6))
    
    # Define Styles for consistency
    styles = {
        'PD':   {'label': 'PD Controller', 'c': 'gray',    'ls': ':',  'lw': 2,   'alpha': 0.5},
        'SBSP': {'label': 'SBSP',          'c': '#ff7f0e', 'ls': '--', 'lw': 1.5, 'alpha': 0.9},
        'ASAC': {'label': 'ASAC',          'c': '#1f77b4', 'ls': '-',  'lw': 1.5, 'alpha': 0.9},
        'DRT':  {'label': 'DRT-RL',        'c': '#2ca02c', 'ls': '-',  'lw': 1.5, 'alpha': 0.9},
        'E2E':  {'label': 'E2E-RL',        'c': '#d62728', 'ls': '-',  'lw': 1.5, 'alpha': 0.9}
    }

    # Iterate through models and plot
    for key, val in data_map.items():
        if val is not None:
            steps, errors = val
            mean_err = np.mean(errors)
            s = styles[key]
            
            # 1. Plot Trajectory with Mean in Label
            # Label format: "Name (μ=0.025m)"
            label_str = f"{s['label']} ($\mu$={mean_err:.3f}m)"
            plt.plot(steps, errors, label=label_str, 
                     color=s['c'], linestyle=s['ls'], linewidth=s['lw'], alpha=s['alpha'])
            
            # 2. Plot Horizontal Mean Line
            # We skip PD line if it's off-screen (>0.15) to keep plot clean
            if mean_err < 0.15:
                plt.axhline(y=mean_err, color=s['c'], linestyle='--', linewidth=1.0, alpha=0.6)

    # Formatting
    plt.title('Control Strategy Comparison: Low Delay Scenario (Focused View)', fontsize=14)
    plt.xlabel('Step', fontsize=12)
    plt.ylabel(r'Tracking Error $\|\mathbf{e}\|$ (m)', fontsize=12)
    
    # Focused Y-axis Limit
    plt.ylim(0, 0.15)
    
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.legend(loc='upper right', fontsize=11, framealpha=0.95)
    plt.tight_layout()
    plt.margins(x=0.01)
    
    # Save
    out_file = 'comparison_low_delay_focused_means.png'
    plt.savefig(out_file, dpi=300)
    print(f"Comparison plot saved successfully to: {out_file}")
    plt.close()

if __name__ == "__main__":
    print("Loading datasets...")
    
    # Load data into a dictionary for easier iteration
    data = {
        'PD':   load_and_calc_error(PATH_PD,   "PD"),
        'SBSP': load_and_calc_error(PATH_SBSP, "SBSP"),
        'ASAC': load_and_calc_error(PATH_ASAC, "ASAC"),
        'DRT':  load_and_calc_error(PATH_DRT,  "DRT-RL"),
        'E2E':  load_and_calc_error(PATH_E2E,  "E2E-RL")
    }
    
    plot_final_comparison(data)

Loading datasets...
Comparison plot saved successfully to: comparison_low_delay_focused_means.png


In [ ]:
import pandas as pd
import numpy as np
import matplotlib

# [FIX] Use non-interactive backend for stability
matplotlib.use('Agg') 
import matplotlib.pyplot as plt

# --- Configuration: File Paths ---
# 1. SBSP
PATH_SBSP = '/media/kai/NewDisk/Kai_thesis/Master_Thesis_E2E_RL_Teleop/libfranka_ws/src/E2E_Teleoperation/E2E_Teleoperation/evaluation/SBSP/high_var_cleaned.csv'

# 2. ASAC
PATH_ASAC = '/media/kai/NewDisk/Kai_thesis/Master_Thesis_E2E_RL_Teleop/libfranka_ws/src/E2E_Teleoperation/E2E_Teleoperation/evaluation/ASAC/high_var.csv'

# 3. PD Controller
PATH_PD   = '/media/kai/NewDisk/Kai_thesis/Master_Thesis_E2E_RL_Teleop/libfranka_ws/src/E2E_Teleoperation/E2E_Teleoperation/evaluation/PD_controller/high_var_cleaned.csv'

# 4. DRT-RL
PATH_DRT  = '/media/kai/NewDisk/Kai_thesis/Master_Thesis_E2E_RL_Teleop/libfranka_ws/src/E2E_Teleoperation/E2E_Teleoperation/evaluation/DRT_RL/high_var_cleaned.csv'

# 5. E2E-RL
PATH_E2E  = '/media/kai/NewDisk/Kai_thesis/Master_Thesis_E2E_RL_Teleop/libfranka_ws/src/E2E_Teleoperation/E2E_Teleoperation/evaluation/data/high_var_cleaned.csv'

# Analysis Settings
MAX_STEPS = 5000
TARGET_DT = 0.02  # Target 50Hz

def load_and_calc_error(file_path, label):
    try:
        df = pd.read_csv(file_path)
        
        # --- Auto-Downsampling Logic ---
        if 'time' in df.columns:
            df = df.sort_values('time').reset_index(drop=True)
            dt_original = df['time'].diff().median()
            
            if dt_original is not None and dt_original > 0:
                step_ratio = int(np.round(TARGET_DT / dt_original))
                if step_ratio > 1:
                    print(f"[{label}] Downsampling by factor {step_ratio}.")
                    df = df.iloc[::step_ratio].reset_index(drop=True)

        # --- Column Validation ---
        required = ['leader_ee_pos_x', 'leader_ee_pos_y', 'leader_ee_pos_z',
                    'follower_ee_pos_x', 'follower_ee_pos_y', 'follower_ee_pos_z']
        
        if not all(col in df.columns for col in required):
            print(f"[{label}] Error: Missing standard columns.")
            return None

        # --- Truncation & Error Calculation ---
        df = df.head(MAX_STEPS)
        leader_pos = df[['leader_ee_pos_x', 'leader_ee_pos_y', 'leader_ee_pos_z']].to_numpy()
        follower_pos = df[['follower_ee_pos_x', 'follower_ee_pos_y', 'follower_ee_pos_z']].to_numpy()
        error_norm = np.linalg.norm(leader_pos - follower_pos, axis=1)
        
        return df.index.to_numpy(), error_norm
        
    except FileNotFoundError:
        print(f"[{label}] Error: File not found.")
        return None
    except Exception as e:
        print(f"[{label}] Unexpected error: {e}")
        return None

def plot_final_comparison(data_map):
    """
    Plots tracking error + average lines + focused view.
    """
    plt.figure(figsize=(12, 6))
    
    # Define Styles for consistency
    styles = {
        'PD':   {'label': 'PD Controller', 'c': 'gray',    'ls': ':',  'lw': 2,   'alpha': 0.5},
        'SBSP': {'label': 'SBSP',          'c': '#ff7f0e', 'ls': '--', 'lw': 1.5, 'alpha': 0.9},
        'ASAC': {'label': 'ASAC',          'c': '#1f77b4', 'ls': '-',  'lw': 1.5, 'alpha': 0.9},
        'DRT':  {'label': 'DRT-RL',        'c': '#2ca02c', 'ls': '-',  'lw': 1.5, 'alpha': 0.9},
        'E2E':  {'label': 'E2E-RL',        'c': '#d62728', 'ls': '-',  'lw': 1.5, 'alpha': 0.9}
    }

    # Iterate through models and plot
    for key, val in data_map.items():
        if val is not None:
            steps, errors = val
            mean_err = np.mean(errors)
            s = styles[key]
            
            # 1. Plot Trajectory with Mean in Label
            # Label format: "Name (μ=0.025m)"
            label_str = f"{s['label']} ($\mu$={mean_err:.3f}m)"
            plt.plot(steps, errors, label=label_str, 
                     color=s['c'], linestyle=s['ls'], linewidth=s['lw'], alpha=s['alpha'])
            
            # 2. Plot Horizontal Mean Line
            # We skip PD line if it's off-screen (>0.15) to keep plot clean
            if mean_err < 0.15:
                plt.axhline(y=mean_err, color=s['c'], linestyle='--', linewidth=1.0, alpha=0.6)

    # Formatting
    plt.title('Control Strategy Comparison: Low Delay Scenario (Focused View)', fontsize=14)
    plt.xlabel('Step', fontsize=12)
    plt.ylabel(r'Tracking Error $\|\mathbf{e}\|$ (m)', fontsize=12)
    
    # Focused Y-axis Limit
    plt.ylim(0, 0.15)
    
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.legend(loc='upper right', fontsize=11, framealpha=0.95)
    plt.tight_layout()
    plt.margins(x=0.01)
    
    # Save
    out_file = 'comparison_low_delay_focused_means.png'
    plt.savefig(out_file, dpi=300)
    print(f"Comparison plot saved successfully to: {out_file}")
    plt.close()

if __name__ == "__main__":
    print("Loading datasets...")
    
    # Load data into a dictionary for easier iteration
    data = {
        'PD':   load_and_calc_error(PATH_PD,   "PD"),
        'SBSP': load_and_calc_error(PATH_SBSP, "SBSP"),
        'ASAC': load_and_calc_error(PATH_ASAC, "ASAC"),
        'DRT':  load_and_calc_error(PATH_DRT,  "DRT-RL"),
        'E2E':  load_and_calc_error(PATH_E2E,  "E2E-RL")
    }
    
    plot_final_comparison(data)

Loading datasets...
Comparison plot saved successfully to: comparison_low_delay_focused_means.png


In [22]:
import pandas as pd
import numpy as np
import matplotlib

# [FIX] Use non-interactive backend for stability
matplotlib.use('Agg') 
import matplotlib.pyplot as plt

# --- Configuration: CORRCTED File Paths (Restored to original) ---
# 1. SBSP (Removed '_traj_')
PATH_SBSP = '/media/kai/NewDisk/Kai_thesis/Master_Thesis_E2E_RL_Teleop/libfranka_ws/src/E2E_Teleoperation/E2E_Teleoperation/evaluation/SBSP/high_var_traj_cleaned.csv'

# 2. ASAC (Removed 'traj_rand_')
PATH_ASAC = '/media/kai/NewDisk/Kai_thesis/Master_Thesis_E2E_RL_Teleop/libfranka_ws/src/E2E_Teleoperation/E2E_Teleoperation/evaluation/ASAC/traj_rand_high_var.csv'

# 3. E2E-RL (Removed '_traj_')
PATH_E2E  = '/media/kai/NewDisk/Kai_thesis/Master_Thesis_E2E_RL_Teleop/libfranka_ws/src/E2E_Teleoperation/E2E_Teleoperation/evaluation/data/high_var_traj_cleaned.csv'

# Analysis Settings
MAX_STEPS = 5000
TARGET_DT = 0.02  # Target 50Hz (Step size 0.02s)

def load_and_calc_error(file_path, label):
    try:
        df = pd.read_csv(file_path)
        
        # --- Auto-Downsampling Logic ---
        if 'time' in df.columns:
            df = df.sort_values('time').reset_index(drop=True)
            dt_original = df['time'].diff().median()
            
            if dt_original is not None and dt_original > 0:
                step_ratio = int(np.round(TARGET_DT / dt_original))
                if step_ratio > 1:
                    print(f"[{label}] Downsampling by factor {step_ratio} (Original dt={dt_original:.4f}s)")
                    df = df.iloc[::step_ratio].reset_index(drop=True)

        # --- Column Validation ---
        required = ['leader_ee_pos_x', 'leader_ee_pos_y', 'leader_ee_pos_z',
                    'follower_ee_pos_x', 'follower_ee_pos_y', 'follower_ee_pos_z']
        
        if not all(col in df.columns for col in required):
            print(f"[{label}] Error: Missing standard columns.")
            return None

        # --- Truncation & Error Calculation ---
        df = df.head(MAX_STEPS)
        leader_pos = df[['leader_ee_pos_x', 'leader_ee_pos_y', 'leader_ee_pos_z']].to_numpy()
        follower_pos = df[['follower_ee_pos_x', 'follower_ee_pos_y', 'follower_ee_pos_z']].to_numpy()
        error_norm = np.linalg.norm(leader_pos - follower_pos, axis=1)
        
        return df.index.to_numpy(), error_norm
        
    except FileNotFoundError:
        print(f"[{label}] Error: File not found.")
        return None
    except Exception as e:
        print(f"[{label}] Unexpected error: {e}")
        return None

def plot_final_comparison(data_map):
    plt.figure(figsize=(12, 6))
    
    # Define Styles (Consistent Colors)
    styles = {
        'SBSP': {'label': 'SBSP',   'c': '#ff7f0e', 'ls': '--', 'lw': 1.5, 'alpha': 0.9},
        'ASAC': {'label': 'ASAC',   'c': '#1f77b4', 'ls': '-',  'lw': 1.5, 'alpha': 0.9},
        'E2E':  {'label': 'E2E-RL', 'c': '#d62728', 'ls': '-',  'lw': 1.5, 'alpha': 0.9}
    }

    # Iterate and Plot
    for key, val in data_map.items():
        if val is not None:
            steps, errors = val
            mean_err = np.mean(errors)
            s = styles[key]
            
            # 1. Trajectory Line
            label_str = f"{s['label']} ($\mu$={mean_err:.3f}m)"
            plt.plot(steps, errors, label=label_str, 
                     color=s['c'], linestyle=s['ls'], linewidth=s['lw'], alpha=s['alpha'])
            
            # 2. Horizontal Mean Line
            # Only plot if visible on scale (< 0.15)
            if mean_err < 0.15: 
                plt.axhline(y=mean_err, color=s['c'], linestyle='--', linewidth=1.0, alpha=0.6)

    # --- Formatting ---
    plt.title('Control Strategy Comparison: High Variance Scenario (Focused View)', fontsize=14)
    plt.xlabel('Step', fontsize=12)
    plt.ylabel(r'Tracking Error $\|\mathbf{e}\|$ (m)', fontsize=12)
    
    # [CRITICAL] Focused Y-axis Limit
    plt.ylim(0, 0.15)
    
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.legend(loc='upper right', fontsize=11, framealpha=0.95)
    
    plt.tight_layout()
    plt.margins(x=0.01)
    
    # Save
    out_file = 'comparison_high_var_focused_3models.png'
    plt.savefig(out_file, dpi=300)
    print(f"Comparison plot saved successfully to: {out_file}")
    plt.close()

if __name__ == "__main__":
    print("Loading datasets...")
    
    # Load ONLY the 3 requested models with CORRECT paths
    data = {
        'SBSP': load_and_calc_error(PATH_SBSP, "SBSP"),
        'ASAC': load_and_calc_error(PATH_ASAC, "ASAC"),
        'E2E':  load_and_calc_error(PATH_E2E,  "E2E-RL")
    }
    
    plot_final_comparison(data)

Loading datasets...
Comparison plot saved successfully to: comparison_high_var_focused_3models.png
